# Streaming Products from S3 During Commissioning

## Introduction

This notebook demonstrates how to search MAST for commissioning data products, and then shows how to stream those data products from MAST into memory without the need to make local copies.

This tutorial has been adapted mainly from the MAST notebook [MAST Metadata Search](https://github.com/spacetelescope/mast_notebooks/blob/roman-prelaunch/notebooks/Roman/MAST_metadata_search/MAST_metadata_search.ipynb) and the Nexus tutorial [Working with ADSF](https://github.com/spacetelescope/roman_notebooks/blob/main/notebooks/working_with_asdf/working_with_asdf.ipynb), with additional information added to support streaming from MAST search results.

## Imports

In [ ]:
import os
import requests
import roman_datamodels as rdm
from astroquery.mast import MastMissions
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from dotenv import load_dotenv

First, we create our `MastMissions` object to act as our gateway to MAST for searches. If you do not have a MAST AUTH token for accessing MAST (needed for Roman OPS access until after commissioning), or if your token has expired, make one now by visiting [MAST.Auth](https://auth.mast.stsci.edu/info).

Once you have a token, you can set it up as an environment variable for later use, or you can store it in a file called `variables.env`. Below, we have assumed the file is in your current working directory (`./`), but the full path to the file can be specified if it is stored elsewhere such as your home directory. The `variables.env` file should look like:

```
MAST_API_TOKEN='yourtokenhere'
```

In [ ]:
_ = load_dotenv(dotenv_path="./variables.env")

# Create MastMissions object and assign mission to 'roman'
missions = MastMissions(mission='roman')

# Login to search and retrieve Roman data
token = os.getenv("MAST_API_TOKEN")    
missions.login(token=token)
               
print(f'Mission: {missions.mission}')
print(f'Service: {missions.service}')

Next, as an example, here we show how to search for data from Program ID 1039 (CAR-190), pass 1, and detector WFI04. Be as specific as you can as the more results MAST returns the longer this cell will take to execute.

In [ ]:
# Make a list of column names to return in the search results. The results will not be ordered
# in this order, so we will re-order later. The fileSetName, which we need for retrieval,
# is not in this list, but will still be returned.
col_list = ['program', 'execution_plan', 'pass', 'segment', 'visit', 'observation', 
            'optical_element', 'exposure_type', 'instrument_name', 'detector', 'productLevel', 
            'product_type', 'exposure_time', 'exposure_start_time', 'exposure_end_time', 'fileSetName']

# Create a dictionary of search criteria
search = {'program': 1039,
         'pass': 1,
         'detector': 'WFI04'}

# Query with column criteria
results = missions.query_criteria(
    **search,
    select_cols=col_list)

# Re-order the column names 
results = results[col_list]

# Display the first 5 results
print(f'Total number of results: {len(results)}')
results[:5]

## Streaming Files from MAST into Memory

Now that we have search results, we can select one and stream it into memory to examine it. You will primarily be interested in the Level 1 (L1) uncalibrated ramps or the Level 2 (L2) calibrated rate images, but there are some additional products as well. See the [WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products) article on RDox for more information.

For the retrieval below, the extension for L1 products is `uncal.asdf`, while for L2 products it is `cal.asdf`.

First, we get the list of data products (files) associated with the search results in the table above. We use the `get_unique_product_list()` method to retrieve only the unique data products.

In [ ]:
products = missions.get_unique_product_list(results[:5])
products[:5]

In [ ]:
filtered = missions.filter_products(products, file_suffix='_cal')
filtered

In [ ]:
tab_index = 3  # Choose which row from the filtered product table you want to retrieve
af = missions.read_product(filtered['filename'][tab_index])

Next we can transform the `AsdfFile` object to a datamodel. This is optional and comes down to how you prefer to interface with the data in the ASDF file.

In [ ]:
dm = rdm.open(af)

## Working with ASDF

For more information on working with ASDF files, see the Nexus tutorial [Working with ADSF](https://github.com/spacetelescope/roman_notebooks/blob/main/notebooks/working_with_asdf/working_with_asdf.ipynb). Briefly, you can access the metadata using dot notation as below:

In [ ]:
dm.meta.observation

From the `observation` section of the metadata, we can see that this does conform to our search for our MRT-7b example (we see it is program ID 114, pass 57, etc.). If we want to plot the data, which for MRT-7b is a test pattern, then we can also do that.

<div class="alert alert-warning" style="color:black; background-color:#ffc5c5; border-color:red;">
    <b>NOTE:</b> If you run the cell below, or in any other way try to access part of the file and get a long exception that ends with "ClientResponseError: 403, message='Forbidden'" followed by a long URI, then try to re-run the missions.read_product() command above. We believe this may be due to a timeout during the streaming, and are investigating.
</div>

In [ ]:
fig, ax = plt.subplots()
norm = simple_norm(dm.data, percent=99.9)
ax.imshow(dm.data, origin='lower', norm=norm)
ax.set_title(filtered[tab_index]['dataset'])
ax.set_ylabel('Science Y (pixels)')
ax.set_xlabel('Science X (pixels)');